<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/IS4490_module8_agent_lab_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4490 - Module 8 - Agents

**Summary:** This notebook demonstrates an extremely simple agent that organizes 3 files based on their contents.

**Task Objective:** The goal here is to see that AI is taking on portions of the task when decisions need to be made. The LLM is given a goal "Organize the files" it then figures out what files need organizing, what organization structure should be used, moves the files to their respective locations and finally completes with a summary back to the user.

**Ingredients:** This is all accomplished with LangChain, Ollama and the Gemma model.

**Repeat Until:** A simple while loop allows for the repetitive behavior needed as the Agent works through its objective.

3 Tools Are given
1) List Directory Tool
2) Read File Tool
3) Move File Tool





## Setups

In [1]:
import json

In [2]:
# Colab setup step 1: install zstd, which the Ollama installer expects on Ubuntu.
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (380 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
# Colab setup step 2: install Ollama.
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Start Ollama

### RESTART ME IF NEEDED

In [4]:
# Colab setup step 3: start the Ollama server and pull the model.
import subprocess
import time

ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

pull = subprocess.run(["ollama", "pull", "gemma4:12b"], capture_output=True, text=True)
print(pull.stdout[-1000:])
print(pull.stderr[-1000:])
print("Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.")


1896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest ⠋ pulling manifest 
pulling 0cb0cd0a5c49:  97% ▕█████████████████ ▏ 170 MB/175 MB  157 MB/s      0s
pulling bb722270d543: 100% ▕█████████████████ ▏ 7.4 GB/7.4 GB  165 MB/s      0s
pulling 7008a656050b:  99% ▕█████████████████ ▏ 462 MB/465 MB  147 MB/s      0s
pulling e5ca99f91896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest ⠙ pulling manifest 
pulling 0cb0cd0a5c49:  97% ▕█████████████████ ▏ 170 MB/175 MB  157 MB/s      0s
pulling bb722270d543: 100% ▕█████████████████ ▏ 7.4 GB/7.4 GB  165 MB/s      0s
pulling 7008a656050b:  99% ▕█████████████████ ▏ 462 MB/465 MB  147 MB/s      0s
pulling e5ca99f91896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest 
writing manifest 
success 

Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.


In [5]:
# Colab setup step 4: install Python packages.
!pip install langchain langchain-ollama --quiet

## Download Gemma Model

In [6]:
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

MODEL = "gemma4:12b"
llm = ChatOllama(model=MODEL, temperature=0)

print(f"Ready. Using {MODEL} through Ollama and LangChain.")

Ready. Using gemma4:12b through Ollama and LangChain.


## Test the LLM with a simple prompt

First time this takes about 2-3 minutes.

In [7]:
llm.invoke("what is the capitol of france?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b', 'created_at': '2026-10-17T02:05:33.811165467Z', 'done': True, 'done_reason': 'stop', 'total_duration': 183966560729, 'load_duration': 120608965776, 'prompt_eval_count': 23, 'prompt_eval_duration': 48097496000, 'eval_count': 55, 'eval_duration': 15252324000, 'logprobs': None, 'model_name': 'gemma4:12b', 'model_provider': 'ollama'}, id='lc_run--01a14798-aa4e-7373-89e3-019443972c18-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 23, 'output_tokens': 55, 'total_tokens': 78})

## Create the Documents

We build 3 documents and directories.


A generic "documents" dir, and the two directories we want to move the files into if they match based on categorization.

In [8]:
!mkdir documents
!mkdir law_documents
!mkdir policy_documents

In [9]:
!echo "This file is a law document." > ./documents/law.txt
!echo "This file is a policy document." > ./documents/policy.txt
!echo "This file is customer email document. it's neither a policy nor law document" > ./documents/unknown.txt

## LLM Tools

1) **List Files** which reads the contents of a directory
2) **Read File** which extracts the contents of a file.
3) **Move File** which moves a file into a directory.

In [10]:
import os
import shutil
import json
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

# --- TOOLS ---

@tool
def list_files(directory: str = "/content/") -> str:
    """List all files inside the specified directory."""
    try:
        files = os.listdir(directory)
        return json.dumps(files)
    except Exception as e:
        return f"Error listing directory {directory}: {str(e)}"

@tool
def read_file(file_path: str) -> str:
    """Read and return the text contents of the given file path."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file {file_path}: {str(e)}"

@tool
def move_file(source_path: str, destination_path: str) -> str:
    """Move a file from source_path to destination_path, creating directories if needed."""
    try:
        dest_dir = os.path.dirname(destination_path)
        shutil.move(source_path, destination_path)
        return f"Successfully moved {source_path} to {destination_path}"
    except Exception as e:
        return f"Error moving {source_path} to {destination_path}: {str(e)}"

# Define the tools list for bind_tools
tools = [list_files, read_file, move_file]
# Map tools for dynamic lookups inside the loop
tools_map = {tool.name: tool for tool in tools}

# User Instructions

This is the user's prompt. The user has requested assistance organizing some files.

In [11]:
# Define your custom instructions / goal for the file-organizing agent
user_instructions = """
I need assistance organizing some files
Organize the .txt files found inside '/content/documents/'
by reading their content, deciding on the type of file they are (law,policy)
and moving law files to '/content/law_documents/'
and policy files to '/content/policy_documents/'
finally summarize the work you did for review.
"""

# Agent System Instructions

In [12]:
system_prompt = """
You are a file-organizing agent.

Your goal is to help the user organize their files.

Guidelines:
1. If you do not know the contents of a directory, call `list_files` first.
2. Run sequentially. If you need to inspect multiple files, read them step by step.
3. Make decisions based on observations inside your memory.
4. When you are finished executing the user's task, output a final summary of what you did and DO NOT call any more tools.
"""

# Agent LOOP

In [13]:
# --- AGENT LOOP ---
from langchain_core.messages import ToolMessage

# Bind the tools natively to the model
llm_with_tools = llm.bind_tools(tools)

# We'll use the native message list to act as our memory
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_instructions)
]

finished = False
max_iterations = 15
iteration = 0

while not finished and iteration < max_iterations:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")

    try:
        # Call the LLM with the current conversation history
        response = llm_with_tools.invoke(messages)

        # Append the AI's response to the memory
        messages.append(response)

        # Check if the model decided to call any tools
        if not response.tool_calls:
            print("Agent completed its organization goal or stopped using tools.")
            print(f"\nFinal Output:\n{response.content}")
            finished = True
            break

        # Execute each tool call requested by the model
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            print(f"Agent Decided to call: {tool_name} with {tool_args}")

            if tool_name in tools_map:
                selected_tool = tools_map[tool_name]

                # Invoke the tool
                tool_result = selected_tool.invoke(tool_args)

                # Create a ToolMessage to report the result back to the model
                tool_message = ToolMessage(
                    content=str(tool_result),
                    name=tool_name,
                    tool_call_id=tool_call["id"]
                )
                messages.append(tool_message)
            else:
                print(f"Unknown tool requested: {tool_name}")

    except Exception as e:
        print(f"Error in iteration: {e}")
        break


--- Iteration 1 ---
Agent Decided to call: list_files with {'directory': '/content/documents/'}

--- Iteration 2 ---
Agent Decided to call: read_file with {'file_path': '/content/documents/law.txt'}

--- Iteration 3 ---
Agent Decided to call: read_file with {'file_path': '/content/documents/unknown.txt'}

--- Iteration 4 ---
Agent Decided to call: read_file with {'file_path': '/content/documents/policy.txt'}

--- Iteration 5 ---
Agent Decided to call: move_file with {'destination_path': '/content/law_documents/law.txt', 'source_path': '/content/documents/law.txt'}

--- Iteration 6 ---
Agent Decided to call: move_file with {'destination_path': '/content/policy_documents/policy.txt', 'source_path': '/content/documents/policy.txt'}

--- Iteration 7 ---
Agent completed its organization goal or stopped using tools.

Final Output:
I have successfully organized the files in `/content/documents/` based on their content. Here is a summary of the actions taken:

1.  **Analyzed files in `/conten

In [14]:
agent_memory

NameError: name 'agent_memory' is not defined